In [1]:
!pip install fastembed qdrant_client

In [2]:
import os
import sys
import re
import uuid
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, SparseVectorParams, SparseIndexParams, PointStruct, SparseVector
from dotenv import load_dotenv

In [3]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [4]:
from google.colab import userdata

qdrant_url = userdata.get("QDRANT_URL")
qdrant_api_key = userdata.get("QDRANT_API_KEY")

if not qdrant_url or not qdrant_api_key:
    print("ERROR: QDRANT_URL and QDRANT_API_KEY must be set in backend/.env")
    sys.exit(1)

collection_name = "mental_health_counseling"

print("Initializing Qdrant Client...")
client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)

Initializing Qdrant Client...


In [5]:
# Recreate collection to support sparse vectors
print(f"Recreating collection '{collection_name}' with dense and sparse vector configurations...")
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    sparse_vectors_config={
        "text": SparseVectorParams(
            index=SparseIndexParams(
                on_disk=False,
            )
        )
    }
)
print("Collection created successfully.")

Recreating collection 'mental_health_counseling' with dense and sparse vector configurations...
Collection created successfully.


In [6]:
print("Loading embedding models...")
print("  - Dense model: all-MiniLM-L6-v2")
dense_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
print("  - Sparse model: prithivida/Splade_PP_en_v1 (SPLADE)")
sparse_model = SparseTextEmbedding(model_name="prithivida/Splade_PP_en_v1")

Loading embedding models...
  - Dense model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  - Sparse model: prithivida/Splade_PP_en_v1 (SPLADE)


In [7]:
  print("Loading dataset Amod/mental_health_counseling_conversations from Hugging Face...")
  dataset = load_dataset("Amod/mental_health_counseling_conversations")
  df = dataset["train"].to_pandas()
  print(f"Loaded {len(df)} raw conversations.")

  print("Cleaning dataset...")
  df = df.dropna().reset_index(drop=True)
  df = df.drop_duplicates().reset_index(drop=True)
  df["Context"] = df["Context"].apply(clean_text)
  df["Response"] = df["Response"].apply(clean_text)
  print(f"Cleaned dataset size: {len(df)}")

Loading dataset Amod/mental_health_counseling_conversations from Hugging Face...
Loaded 3512 raw conversations.
Cleaning dataset...
Cleaned dataset size: 2752


In [8]:
contexts = df["Context"].tolist()
responses = df["Response"].tolist()

In [9]:
# print("Generating dense embeddings...")
# dense_embeddings = dense_model.encode(contexts, batch_size=128, show_progress_bar=True)


In [10]:
# print("Generating sparse embeddings...")
# # SparseTextEmbedding.embed returns a generator of SparseEmbedding objects
# sparse_embeddings_gen = sparse_model.embed(contexts)
# sparse_embeddings = list(sparse_embeddings_gen)

In [11]:
BATCH_SIZE = 100 # Adjust this down to 100 if you still hit memory limits
total_batches = (len(contexts) // BATCH_SIZE) + 1

print(f"Starting batch processing. Total batches: {total_batches}")

# Process the dataset in chunks
for i in range(0, len(contexts), BATCH_SIZE):
    batch_contexts = contexts[i:i+BATCH_SIZE]
    batch_responses = responses[i:i+BATCH_SIZE]

    print(f"Processing batch {(i//BATCH_SIZE) + 1} of {total_batches}...")

    # 1. Generate embeddings just for this small batch
    batch_dense = dense_model.encode(batch_contexts, show_progress_bar=False)
    batch_sparse = list(sparse_model.embed(batch_contexts))

    # 2. Build the Qdrant PointStructs
    points = []
    for ctx, rsp, dense_emb, sparse_emb in zip(batch_contexts, batch_responses, batch_dense, batch_sparse):
        qdrant_sparse = SparseVector(
            indices=sparse_emb.indices.tolist() if hasattr(sparse_emb.indices, "tolist") else list(sparse_emb.indices),
            values=sparse_emb.values.tolist() if hasattr(sparse_emb.values, "tolist") else list(sparse_emb.values)
        )

        points.append(
            PointStruct(
                id=str(uuid.uuid4()),
                vector={
                    "": dense_emb.tolist(),
                    "text": qdrant_sparse
                },
                payload={
                    "context": ctx,
                    "response": rsp
                }
            )
        )

    # 3. Upload immediately, then let Python's garbage collector clear the batch RAM
    client.upload_points(
        collection_name=collection_name,
        points=points
    )
print("Ingestion completed successfully!")

Starting batch processing. Total batches: 28
Processing batch 1 of 28...
Processing batch 2 of 28...
Processing batch 3 of 28...
Processing batch 4 of 28...
Processing batch 5 of 28...
Processing batch 6 of 28...
Processing batch 7 of 28...
Processing batch 8 of 28...
Processing batch 9 of 28...
Processing batch 10 of 28...
Processing batch 11 of 28...
Processing batch 12 of 28...
Processing batch 13 of 28...
Processing batch 14 of 28...
Processing batch 15 of 28...
Processing batch 16 of 28...
Processing batch 17 of 28...
Processing batch 18 of 28...
Processing batch 19 of 28...
Processing batch 20 of 28...
Processing batch 21 of 28...
Processing batch 22 of 28...
Processing batch 23 of 28...
Processing batch 24 of 28...
Processing batch 25 of 28...
Processing batch 26 of 28...
Processing batch 27 of 28...
Processing batch 28 of 28...
Ingestion completed successfully!


In [13]:
collections = client.get_collections()
print(collections)

collections=[CollectionDescription(name='mental_health_counseling')]


In [17]:
# 1. Retrieve the lightweight list of collections
collections_response = client.get_collections()

# 2. Iterate through the description objects to extract the names
for collection_desc in collections_response.collections:
    collection_name = collection_desc.name

    # 3. Fetch the deep architectural details for the specific collection
    info = client.get_collection(collection_name)

    print(f"\n=== Collection: {collection_name} ===")

    # Check cluster health and data volume
    print(f"Status: {info.status}")
    print(f"Total Points: {info.points_count}")
    print(f"Indexed Vectors: {info.indexed_vectors_count}")

    # Access the vector configuration (size and distance metrics)
    # The structure differs slightly depending on if you use one vector or multiple named vectors
    vectors_config = info.config.params.vectors

    if hasattr(vectors_config, 'size'):
        # Standard single-vector configuration
        print(f"Vector Size: {vectors_config.size}")
        print(f"Distance Metric: {vectors_config.distance}")
    else:
        # Multi-vector configuration (e.g., Hybrid Search with dense and sparse vectors)
        for vec_name, vec_params in vectors_config.items():
            print(f"Named Vector '{vec_name}':")
            if hasattr(vec_params, 'size'):
                 print(f"  - Size: {vec_params.size}, Distance: {vec_params.distance}")
            else:
                 print(f"  - Type: Sparse Vector")


=== Collection: mental_health_counseling ===
Status: green
Total Points: 2752
Indexed Vectors: 2752
Vector Size: 384
Distance Metric: Cosine


In [19]:

from qdrant_client.models import TextIndexParams, TextIndexType, TokenizerType



# Instruct Qdrant to build a text index on the "context" payload field
client.create_payload_index(
    collection_name="mental_health_counseling",
    field_name="context",
    field_schema=TextIndexParams(
        type=TextIndexType.TEXT,
        tokenizer=TokenizerType.WORD,
        min_token_len=2,
        max_token_len=15,
        lowercase=True,
    )
)
print("Payload index created successfully.")

Payload index created successfully.
